# Combat Critic — Colab v1 formal train (tip #4)

**Data:** full colab_v1 `transitions.npz` (~500k combat rows, obs 181).  
**Init:** warm-start **`combat_critic_smoke.pt`** only (not bh_v1 PPO).  
**Scale:** ≥3 epochs over full train split **and** ≥1.5M sample-updates (Lab lock).  
**Out:** `combat_critic_colab_v1.pt` + `combat_critic_colab_v1.onnx` on Drive.  
**Gates:** finite train/val loss; val vs smoke baseline ratio; ONNX max_abs_err ≤ **1e-5**.

No game / EP / dll / Jev. Not tip#5 Offline RL.

In [ ]:
# !git clone --depth 1 https://github.com/EienteiPharma/sts2-rl-agent.git /content/sts2-rl-agent
!pip install -q numpy torch onnx onnxruntime

import sys
from pathlib import Path

REPO = Path("/content/sts2-rl-agent")
if not REPO.is_dir():
    raise SystemExit("Clone repo to /content/sts2-rl-agent first (uncomment git clone).")
sys.path.insert(0, str(REPO))

In [ ]:
from google.colab import drive  # type: ignore
from pathlib import Path

DRIVE_COLAB = "/content/drive/MyDrive/sts2/colab"

drive.mount("/content/drive")
Path(DRIVE_COLAB).mkdir(parents=True, exist_ok=True)

FEATURES_PATH = f"{DRIVE_COLAB}/runenv_combat_buffer_colab_v1/transitions.npz"
INIT_CKPT = f"{DRIVE_COLAB}/combat_critic_smoke.pt"
OUT_CKPT = f"{DRIVE_COLAB}/combat_critic_colab_v1.pt"
OUT_ONNX = f"{DRIVE_COLAB}/combat_critic_colab_v1.onnx"

for p in (FEATURES_PATH, INIT_CKPT):
    if not Path(p).is_file():
        raise SystemExit(f"Missing input: {p}")

print("FEATURES_PATH", FEATURES_PATH)
print("INIT_CKPT", INIT_CKPT)
print("OUT_CKPT", OUT_CKPT)
print("OUT_ONNX", OUT_ONNX)

In [ ]:
from sts2_env.colab.combat_critic import (
    CriticColabV1TrainConfig,
    DEFAULT_ONNX_VERIFY_COLAB_V1,
    export_critic_onnx,
    resolve_colab_v1_epochs,
    train_critic_colab_v1,
    verify_critic_onnx,
)
import numpy as np

cfg = CriticColabV1TrainConfig(batch_size=1024, lr=3e-4, seed=0)
with np.load(FEATURES_PATH, allow_pickle=False) as z:
    n_rows = int(z["obs"].shape[0])
n_val = max(1, int(round(n_rows * cfg.val_frac)))
n_train = n_rows - n_val
epochs, spe = resolve_colab_v1_epochs(n_train, cfg.batch_size)
print("n_rows", n_rows, "epochs_planned", epochs, "steps_per_epoch", spe)

meta = train_critic_colab_v1(
    FEATURES_PATH,
    OUT_CKPT,
    INIT_CKPT,
    config=cfg,
)
meta

In [ ]:
import numpy as np

assert meta["sample_updates"] >= 1_500_000
assert meta["epochs"] >= 3
assert all(np.isfinite(meta["train_losses"]))
assert all(np.isfinite(meta["val_losses"]))

onnx_out = export_critic_onnx(OUT_CKPT, OUT_ONNX)
verify = verify_critic_onnx(
    OUT_CKPT,
    onnx_out,
    max_abs_err=DEFAULT_ONNX_VERIFY_COLAB_V1,
)
print("CRITIC_COLAB_V1_PASS", OUT_CKPT, OUT_ONNX, verify["max_abs_err"])
verify